# Faza 3 - pełna generacja wariantu D (10 klas × N obrazów)

Cel: wygenerować syntetyki dla wszystkich 10 klas, proste prompty, bez LoRA. Po każdej klasie automatyczny push do HF Hub - jeśli Colab padnie, postęp jest zachowany.

**Przed uruchomieniem:** Runtime -> Change runtime type -> **T4 GPU**, Secrets -> `HF_TOKEN`.

## Workflow

1. **Pull** istniejącego `variant_D/` z HF (jeśli wznawiasz po przerwaniu)
2. **Loop po klasach:** policz ile już jest → wygeneruj brakujące → push tej klasy do HF
3. **Summary** - co zostało wygenerowane

Na dysku:
- `data/synthetic/variant_D/<breed>/<breed>_NNNN.png` × `N_PER_CLASS` per klasa
- `data/synthetic/variant_D/generate_*_metrics.csv` - log generacji

Na HF Hub (incrementalny upload):
- `variant_D/<breed>/` po każdej ukończonej klasie

## Czas

T4: ~7-8 s/obraz × N × 10 klas. Dla N=300: ~6-7h total.

## 1. Klonowanie repo (Colab)

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls

## 2. Instalacja

In [ ]:
!pip install -q diffusers==0.31.0 accelerate==1.1.1 transformers==4.46.3 timm==1.0.11 PyYAML==6.0.2 huggingface_hub==0.26.2

## 3. HF login

In [ ]:
from huggingface_hub import login, whoami

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('brak HF_TOKEN - ustaw w Colab Secrets')

login(token=HF_TOKEN, add_to_git_credential=False)
print('HF logged in as:', whoami()['name'])

## 4. Konfiguracja

Zmień `N_PER_CLASS` i `HF_REPO_ID` na swoje wartości.

In [ ]:
from pathlib import Path

HF_REPO_ID = 'micwuj/dlicv-synth'
N_PER_CLASS = 300
BREEDS_TO_GENERATE = None  # None = wszystkie 10, albo np. ['Ragdoll', 'Birman']

VARIANT = 'D'
OUTPUT_ROOT = Path('data/synthetic') / f'variant_{VARIANT}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('output:', OUTPUT_ROOT.resolve())

## 5. Pull istniejącego `variant_D/` z HF (resume)

Jeśli wznawiasz po przerwaniu - pobiera to co już jest na HF.

In [ ]:
from huggingface_hub import HfApi, snapshot_download, create_repo

create_repo(HF_REPO_ID, repo_type='dataset', exist_ok=True, private=True)
try:
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type='dataset',
        allow_patterns=f'variant_{VARIANT}/**',
        local_dir='data/synthetic',
    )
    print('pulled existing variant_D from HF')
except Exception as e:
    print(f'nothing to pull yet (ok): {e}')

for d in sorted(OUTPUT_ROOT.glob('*')):
    if d.is_dir():
        n = len(list(d.glob('*.png')))
        print(f'  {d.name}: {n} imgs')

## 6. Pobranie Pets (do ewentualnego sanity check, opcjonalne)

In [ ]:
PETS_IMAGES = Path('data/raw/oxford-iiit-pet/images')
if PETS_IMAGES.exists() and any(PETS_IMAGES.iterdir()):
    print(f'present: {PETS_IMAGES}')
else:
    print('skip Pets download (uncomment if needed)')
    !python scripts/download_pets.py

## 7. Setup: device, pipeline, helpers

In [ ]:
import time

from src.synth.generate import GenerateConfig, build_pipeline, generate_for_breed
from src.synth.prompts import BREEDS, iter_prompt_variations
from src.utils.device import device_info, get_device
from src.utils.logger import CSVLogger

device = get_device()
print('device:', device_info(device))

cfg = GenerateConfig(image_size=512, num_inference_steps=30, guidance_scale=7.5, batch_size=4)
pipe = build_pipeline(cfg, device)
print('pipeline ready')

In [ ]:
def count_existing(breed: str) -> int:
    breed_dir = OUTPUT_ROOT / breed
    if not breed_dir.exists():
        return 0
    return len(list(breed_dir.glob(f'{breed}_*.png')))

def build_pairs(breed: str, n: int, start_idx: int) -> tuple[list[str], list[int]]:
    pool = iter_prompt_variations(breed, mode='simple')
    prompts = [pool[(start_idx + i) % len(pool)] for i in range(n)]
    seeds = [start_idx + i for i in range(n)]
    return prompts, seeds

def upload_breed_to_hf(breed: str, api: HfApi) -> str:
    breed_dir = OUTPUT_ROOT / breed
    path_in_repo = f'variant_{VARIANT}/{breed}'
    api.upload_folder(
        folder_path=str(breed_dir),
        path_in_repo=path_in_repo,
        repo_id=HF_REPO_ID,
        repo_type='dataset',
        commit_message=f'variant {VARIANT}: {breed} ({len(list(breed_dir.glob("*.png")))} imgs)',
    )
    return f'https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/{path_in_repo}'

## 8. Generacja per klasa + push do HF

In [ ]:
api = HfApi()
logger = CSVLogger(OUTPUT_ROOT, run_name=f'generate_{VARIANT}_{int(time.time())}')
breeds = BREEDS_TO_GENERATE or list(BREEDS.keys())

total_start = time.time()
for i, breed in enumerate(breeds, 1):
    existing = count_existing(breed)
    to_generate = max(0, N_PER_CLASS - existing)
    print(f'\n[{i}/{len(breeds)}] {breed}: have {existing}, need {to_generate}')
    if to_generate == 0:
        print(f'  skip (complete)')
        continue

    prompts, seeds = build_pairs(breed, to_generate, start_idx=existing)
    breed_dir = OUTPUT_ROOT / breed
    gen_start = time.time()
    generate_for_breed(pipe, breed, prompts, seeds, breed_dir, cfg, device, logger)
    gen_elapsed = time.time() - gen_start
    print(f'  generated {to_generate} in {gen_elapsed:.0f}s ({gen_elapsed/to_generate:.1f}s/img)')

    up_start = time.time()
    url = upload_breed_to_hf(breed, api)
    print(f'  uploaded ({time.time()-up_start:.0f}s) -> {url}')

    total_elapsed = (time.time() - total_start) / 60
    print(f'  [progress] total elapsed: {total_elapsed:.1f} min')

print(f'\nDONE: {(time.time()-total_start)/60:.1f} min total')

## 9. Summary

In [ ]:
print(f'variant_{VARIANT}/ contents on disk:')
total = 0
for d in sorted(OUTPUT_ROOT.glob('*')):
    if d.is_dir():
        n = len(list(d.glob('*.png')))
        total += n
        marker = 'OK' if n >= N_PER_CLASS else 'incomplete'
        print(f'  {d.name}: {n}/{N_PER_CLASS}  [{marker}]')
print(f'total: {total} images')
print(f'\nHF dataset: https://huggingface.co/datasets/{HF_REPO_ID}')